In [ ]:
import sympy as sym
import numpy as np
import dill

sym.init_printing(use_unicode=True)

print(np.__version__)

2.5.2


# Intro

## Preliminaries

\begin{align*}
q\{\phi \} &\triangleq \text{Exp}(\phi) = \begin{bmatrix} \cos(\frac{||\phi||}{2}) \\ \frac{\phi}{||\phi||} \sin(\frac{||\phi||}{2}) \end{bmatrix} \approx \begin{bmatrix} 1 \\ \frac{\phi}{2} \end{bmatrix} \text{ for small angles.} \\
\\
[\omega]_\times &\triangleq \begin{bmatrix}
0 & -\omega_z & \omega_y \\
\omega_z & 0 & -\omega_x \\
-\omega_y & \omega_x & 0
\end{bmatrix}
\end{align*}

In [ ]:
def quat_from_axis_angle(Phi, small_angle=True):
    if small_angle:
        return sym.Matrix([1, *(0.5 * Phi)])
    
    norm = sym.sqrt(Phi[0]**2 + Phi[1]**2 + Phi[2]**2)
    return sym.Matrix([
        sym.cos(0.5 * norm),
        *(Phi * (1 / norm) * sym.sin(0.5 * norm))
    ])

def skew_symmetric_matrix(omega):
    omega_x, omega_y, omega_z = omega[0], omega[1], omega[2]
    return sym.Matrix([
        [0, -omega_z, omega_y],
        [omega_z, 0, -omega_x],
        [-omega_y, omega_x, 0]
    ])

## Helper Functions

In [ ]:
# rotation matrix from quaternion
def R(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    return sym.Matrix([
        [1 - 2*(qy**2 + qz**2), 2*(qx*qy - qz*qw),     2*(qx*qz + qy*qw)],
        [2*(qx*qy + qz*qw),     1 - 2*(qx**2 + qz**2), 2*(qy*qz - qx*qw)],
        [2*(qx*qz - qy*qw),     2*(qy*qz + qx*qw),     1 - 2*(qx**2 + qy**2)]
    ])

def quat_inv(q):
    return sym.Matrix([q[0], -q[1], -q[2], -q[3]])
    
def quat_norm(q):
    return q / sym.sqrt(q[0]**2 + q[1]**2 + q[2]**2 + q[3]**2)


def left_quat_matrix(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    
                            #w
                            #x
                            #y
                            #z
    return sym.Matrix([
        [qw, -qx, -qy, -qz], 
        [qx,  qw, -qz,  qy], 
        [qy,  qz,  qw, -qx], 
        [qz, -qy,  qx,  qw]  
    ])

# def right_quat_matrix(q):
#     qw, qx, qy, qz = q[0], q[1], q[2], q[3]
#     return sym.Matrix([
#         [qw, -qx, -qy, -qz], 
#         [qx,  qw,  qz, -qy], 
#         [qy, -qz,  qw,  qx], 
#         [qz,  qy, -qx,  qw]  
#     ])
# 
# def rotate_point(point, q):
#     return sym.Matrix((
#         left_quat_matrix(q) * right_quat_matrix(quat_inv(q)) * sym.Matrix([0, *point])
#     )[1:4])

## Additional Variables

For each cycle, we calculate $\Delta t$ in code and we assume a fixed gravity acceleration force. For the offset between IMU and camera, we define translation and rotation.

\begin{align*}
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

In [ ]:
dt = sym.Symbol('\\Delta t')
gravity = sym.Symbol('g')

# [0, -0.0013, -0.00662]
IMU_to_cam_translation = sym.Matrix(sym.symbols("\\Delta{p}_x, \\Delta{p}_y, \\Delta{p}_z"))
# Phi = 102 * (constants.pi / 180) 
angle = sym.Symbol("\\phi")
IMU_to_cam_rotation = sym.Matrix([sym.cos(-angle / 2), sym.sin(-angle / 2), 0, 0])

dt, gravity, IMU_to_cam_translation, IMU_to_cam_rotation

⎛                            ⎡   ⎛\phi⎞ ⎤⎞
⎜                            ⎢cos⎜────⎟ ⎥⎟
⎜                            ⎢   ⎝ 2  ⎠ ⎥⎟
⎜             ⎡\Delta{p}ₓ ⎤  ⎢          ⎥⎟
⎜             ⎢           ⎥  ⎢    ⎛\phi⎞⎥⎟
⎜\Delta t, g, ⎢\Delta{p}_y⎥, ⎢-sin⎜────⎟⎥⎟
⎜             ⎢           ⎥  ⎢    ⎝ 2  ⎠⎥⎟
⎜             ⎣\Delta{p}_z⎦  ⎢          ⎥⎟
⎜                            ⎢    0     ⎥⎟
⎜                            ⎢          ⎥⎟
⎝                            ⎣    0     ⎦⎠

---
# Input

\begin{align*}
&\text{Accelerometer input: } &\tilde a && &\text{noise: } &\tilde a_n \\
&\text{Gyroscope input: } &\tilde \omega && &\text{noise: } &\tilde \omega_n\\
\end{align*}

In [ ]:
a_tilde = sym.Matrix(sym.symbols("\\tilde{a}_x, \\tilde{a}_y, \\tilde{a}_z"))
omega_tilde = sym.Matrix(sym.symbols("\\tilde{\\omega}_x, \\tilde{\\omega}_y, \\tilde{\\omega}_z"))

u_tilde = sym.Matrix.vstack(a_tilde, omega_tilde)

a_tilde_n = sym.Matrix(sym.symbols("\\tilde{a}_{n\\,x}, \\tilde{a}_{n\\,y}, \\tilde{a}_{n\\,z}"))
omega_tilde_n = sym.Matrix(sym.symbols("\\tilde{\\omega}_{n\\,x}, \\tilde{\\omega}_{n\\,y}, \\tilde{\\omega}_{n\\,z}"))

u_tilde_n = sym.Matrix.vstack(a_tilde_n, omega_tilde_n)

u_tilde.T, u_tilde_n.T

([\tilde{a}ₓ  \tilde{a}_y  \tilde{a}_z  \tilde{\omega}ₓ  \tilde{\omega}_y  \ti ↪

↪ lde{\omega}_z], [\tilde{a}_{n,x}  \tilde{a}_{n,y}  \tilde{a}_{n,z}  \tilde{\ ↪

↪ omega}_{n,x}  \tilde{\omega}_{n,y}  \tilde{\omega}_{n,z}])

---
# Nominal State

## IMU

\begin{align*}
x_\text{IMU} = \begin{bmatrix} p & v & q & a_b & \omega_b \end{bmatrix}^\top \qquad \qquad x \in SO(3) \times \left(\mathbb{R}^3\right)^4
\end{align*}

In [ ]:
p = sym.Matrix(sym.symbols("p_x, p_y, p_z"))                                           # position
v = sym.Matrix(sym.symbols("v_x, v_y, v_z"))                                           # velocity
q = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))                                      # orientation (scalar first)
a_b = sym.Matrix(sym.symbols("a_{b\\,x}, a_{b\\,y}, a_{b\\,z}"))                       # accelerometer bias
omega_b = sym.Matrix(sym.symbols("\\omega_{b\\,x}, \\omega_{b\\,y}, \\omega_{b\\,z}")) # gyroscope bias

x_IMU = sym.Matrix.vstack(p, v, q, a_b, omega_b)
x_IMU.T

[pₓ  p_y  p_z  vₓ  v_y  v_z  q_w  qₓ  q_y  q_z  a_{b,x}  a_{b,y}  a_{b,z}  \om ↪

↪ ega_{b,x}  \omega_{b,y}  \omega_{b,z}]

### Kinematics

$$
\begin{align}
p &\leftarrow p + v\Delta t + \frac{1}{2}( \mathbf{R}^{\{q\}} (\tilde a - a_b) + g)\Delta t^2 \\
v &\leftarrow v + (\mathbf{R}^{\{q\}}(\tilde a - a_b) + g)\Delta t \\
q &\leftarrow q \otimes q\{(\tilde \omega - \omega_b) \Delta t\}\\
a_b &\leftarrow a_b \\
\omega_b &\leftarrow \omega_b 
\end{align}
$$

In [ ]:
f_p = p + v * dt + 0.5 * (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt**2
f_v = v + (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt
f_q = left_quat_matrix(q) * quat_from_axis_angle((omega_tilde - omega_b) * dt)
f_a_b = a_b
f_omega_b = omega_b

f_nominal_state_IMU = sym.Matrix.vstack(f_p, f_v, f_q, f_a_b, f_omega_b)

f_nominal_state_IMU

⎡            2 ⎛                           ⎛       2        2    ⎞             ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ + 0.5⋅(\til ↪
⎢                                                                              ↪
⎢            2 ⎛                                                               ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(2⋅q_w⋅q_z + 2⋅qₓ⋅q_y) + 0.5⋅(\tild ↪
⎢                                                                              ↪
⎢        2 ⎛                                                                   ↪
⎢\Delta t ⋅⎝0.5⋅g + 0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z) + 0.5⋅( ↪
⎢                                                                              ↪
⎢                           ⎛                       ⎛       2        2    ⎞    ↪
⎢                  \Delta t⋅⎝(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ +  ↪
⎢                                                                              ↪
⎢                           

## Speakers

\begin{align*}
x_s = \begin{bmatrix} p_i & v_i & q_i \end{bmatrix}^\top \qquad \qquad x_s \in \prod_{i = 1}^n \left(SO(3) \times \left(\mathbb{R}^3\right)^2 \right)
\end{align*}

In [ ]:
p_i = sym.Matrix(sym.symbols("p_{i\\,x}, p_{i\\,y}, p_{i\\,z}"))
v_i = sym.Matrix(sym.symbols("v_{i\\,x}, v_{i\\,y}, v_{i\\,z}"))
q_i = sym.Matrix(sym.symbols("q_{i\\,w}, q_{i\\,x}, q_{i\\,y}, q_{i\\,z}"))

x_speaker = sym.Matrix.vstack(p_i, v_i, q_i)
x_speaker.T

[p_{i,x}  p_{i,y}  p_{i,z}  v_{i,x}  v_{i,y}  v_{i,z}  q_{i,w}  q_{i,x}  q_{i, ↪

↪ y}  q_{i,z}]

### Kinematics

\begin{align*}
p_i &\leftarrow p_i + v_i \Delta t \\
\text{Discrete Case} \qquad \qquad v_i &\leftarrow v_i \\
q_i &\leftarrow q_i \\
\end{align*}

In [ ]:
f_p_i = p_i + v_i * dt
f_v_i = v_i
f_q_i = q_i

f_nominal_state_speaker = sym.Matrix.vstack(f_p_i, f_v_i, f_q_i)
# f_nominal_state_speaker

## Combined State

\begin{align*}
x = \begin{bmatrix} p & v & q & a_b & \omega_b & p_i & v_i & q_i & p_{i + 1} & v_{i + 1} & q_{i + 1} & \cdots \end{bmatrix}^\top
\end{align*}

In [ ]:
x = sym.Matrix.vstack(x_IMU, x_speaker)
f_nominal_state = sym.Matrix.vstack(f_nominal_state_IMU, f_nominal_state_speaker)

x.T

[pₓ  p_y  p_z  vₓ  v_y  v_z  q_w  qₓ  q_y  q_z  a_{b,x}  a_{b,y}  a_{b,z}  \om ↪

↪ ega_{b,x}  \omega_{b,y}  \omega_{b,z}  p_{i,x}  p_{i,y}  p_{i,z}  v_{i,x}  v ↪

↪ _{i,y}  v_{i,z}  q_{i,w}  q_{i,x}  q_{i,y}  q_{i,z}]

---
# Error State

## IMU

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b \end{bmatrix}^\top \qquad \qquad x \in \left(\mathbb{R}^3\right)^5
\end{align*}

In [ ]:
delta_p = sym.Matrix(sym.symbols("\\delta{p}_x, \\delta{p}_y, \\delta{p}_z"))
delta_v = sym.Matrix(sym.symbols("\\delta{v}_x, \\delta{v}_y, \\delta{v}_z"))
delta_theta = sym.Matrix(sym.symbols("\\delta{\\theta}_x, \\delta{\\theta}_y, \\delta{\\theta}_z"))
delta_a_b = sym.Matrix(sym.symbols("\\delta{a}_{b\\,x}, \\delta{a}_{b\\,y}, \\delta{a}_{b\\,z}"))
delta_omega_b = sym.Matrix(sym.symbols("\\delta{\\omega}_{b\\,x}, \\delta{\\omega}_{b\\,y}, \\delta{\\omega}_{b\\,z}"))

delta_x_IMU = sym.Matrix.vstack(delta_p, delta_v, delta_theta, delta_a_b, delta_omega_b)

delta_v_n = sym.Matrix(sym.symbols("\\delta{v}_{n\\,x}, \\delta{v}_{n\\,y}, \\delta{v}_{n\\,z}"))
delta_theta_n = sym.Matrix(sym.symbols("\\delta{\\theta}_{n\\,x}, \\delta{\\theta}_{n\\,y}, \\delta{\\theta}_{n\\,z}"))
delta_a_n = sym.Matrix(sym.symbols("\\delta{a}_{n\\,x}, \\delta{a}_{n\\,y}, \\delta{a}_{n\\,z}"))
delta_omega_n = sym.Matrix(sym.symbols("\\delta{\\omega}_{n\\,x}, \\delta{\\omega}_{n\\,y}, \\delta{\\omega}_{n\\,z}"))

delta_x_n_IMU = sym.Matrix.vstack(delta_v_n, delta_theta_n, delta_a_n, delta_omega_n)

delta_x_IMU.T, delta_x_n_IMU.T

([\delta{p}ₓ  \delta{p}_y  \delta{p}_z  \delta{v}ₓ  \delta{v}_y  \delta{v}_z   ↪

↪ \delta{\theta}ₓ  \delta{\theta}_y  \delta{\theta}_z  \delta{a}_{b,x}  \delta ↪

↪ {a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delta{\omega}_{b,y}  \del ↪

↪ ta{\omega}_{b,z}], [\delta{v}_{n,x}  \delta{v}_{n,y}  \delta{v}_{n,z}  \delt ↪

↪ a{\theta}_{n,x}  \delta{\theta}_{n,y}  \delta{\theta}_{n,z}  \delta{a}_{n,x} ↪

↪   \delta{a}_{n,y}  \delta{a}_{n,z}  \delta{\omega}_{n,x}  \delta{\omega}_{n, ↪

↪ y}  \delta{\omega}_{n,z}])

### Kinematics

\begin{align*}
\delta p &\leftarrow \delta p + \delta v\Delta t & \delta a_b &\leftarrow \delta a_b + \delta a_n \\
\text{Discrete Case} \qquad \qquad \delta v &\leftarrow \delta v + (-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n} \qquad & \delta \omega_b &\leftarrow \delta \omega_b + \delta \omega_n \\
\delta \theta &\leftarrow (\mathbf{R}^{\{(\tilde \omega - \omega_b) \Delta t\}})^\top \delta \theta - \delta \omega_b \Delta t + \delta \theta_{n}
\end{align*}

In [ ]:
f_delta_p = delta_p + delta_v * dt
f_delta_v = delta_v + (- R(q) * skew_symmetric_matrix(a_tilde - a_b) * delta_theta \
    - R(q) * delta_a_b) * dt + delta_v_n
f_delta_theta = R(quat_from_axis_angle((omega_tilde - omega_b) * dt)).T * delta_theta \
    - delta_omega_b * dt + delta_theta_n
f_delta_a_b = delta_a_b + delta_a_n
f_delta_omega_b = delta_omega_b + delta_omega_n

f_error_state_IMU = sym.Matrix.vstack(f_delta_p, f_delta_v, f_delta_theta, f_delta_a_b, f_delta_omega_b)

# print(sym.latex(f_error_state_transition))
f_error_state_IMU

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢         ⎛                                                                    ↪
⎢\Delta t⋅⎝\delta{\theta}ₓ⋅((-\tilde{a}_y + a_{b,y})⋅(-2⋅q_w⋅q_y - 2⋅qₓ⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                ⎛                                                  ↪
⎢ \Delta t⋅⎝\delta{\theta}ₓ⋅⎝(-\tilde{a}_y + a_{b,y})⋅(2⋅q_w⋅qₓ - 2⋅q_y⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                

## Speakers

\begin{align*}
\delta x_s = \begin{bmatrix} \delta p_i & \delta v_i & \delta \theta_i \end{bmatrix}^\top \qquad \qquad x_s \in \left(\mathbb{R}^3\right)^3
\end{align*}

In [ ]:
delta_p_i = sym.Matrix(sym.symbols("\\delta{p}_{i\\,x}, \\delta{p}_{i\\,y}, \\delta{p}_{i\\,z}"))
delta_v_i = sym.Matrix(sym.symbols("\\delta{v}_{i\\,x}, \\delta{v}_{i\\,y}, \\delta{v}_{i\\,z}"))
delta_theta_i = sym.Matrix(sym.symbols("\\delta{\\theta}_{i\\,x}, \\delta{\\theta}_{i\\,y}, \\delta{\\theta}_{i\\,z}"))

delta_x_speaker = sym.Matrix.vstack(delta_p_i, delta_v_i, delta_theta_i)
delta_x_speaker.T


[\delta{p}_{i,x}  \delta{p}_{i,y}  \delta{p}_{i,z}  \delta{v}_{i,x}  \delta{v} ↪

↪ _{i,y}  \delta{v}_{i,z}  \delta{\theta}_{i,x}  \delta{\theta}_{i,y}  \delta{ ↪

↪ \theta}_{i,z}]

### Kinematics

\begin{align*}
\delta p_i &\leftarrow  \\
\text{Discrete Case} \qquad \qquad \delta v_i &\leftarrow \\
\delta q_i &\leftarrow \\
\end{align*}

In [ ]:
# TODO

## Combined State

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b & \delta p_i & \delta v_i & \delta \theta_i & \delta p_{i + 1} & \delta v_{i + 1} & \delta \theta_{i + 1} & \cdots \end{bmatrix}^\top \\
\end{align*}

In [ ]:
delta_x = sym.Matrix.vstack(delta_x_IMU, delta_x_speaker)

delta_x.T

[\delta{p}ₓ  \delta{p}_y  \delta{p}_z  \delta{v}ₓ  \delta{v}_y  \delta{v}_z  \ ↪

↪ delta{\theta}ₓ  \delta{\theta}_y  \delta{\theta}_z  \delta{a}_{b,x}  \delta{ ↪

↪ a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delta{\omega}_{b,y}  \delt ↪

↪ a{\omega}_{b,z}  \delta{p}_{i,x}  \delta{p}_{i,y}  \delta{p}_{i,z}  \delta{v ↪

↪ }_{i,x}  \delta{v}_{i,y}  \delta{v}_{i,z}  \delta{\theta}_{i,x}  \delta{\the ↪

↪ ta}_{i,y}  \delta{\theta}_{i,z}]

---
# Measurements

## IMU

$$
h_{IMU} = \mathbf{R}^{\{q\}}g + a_b
$$

In [ ]:
h_IMU = R(q).T * sym.Matrix([0, 0, gravity]) + a_b
h_IMU

⎡a_{b,x} + g⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z)⎤
⎢                                   ⎥
⎢a_{b,y} + g⋅(2⋅q_w⋅qₓ + 2⋅q_y⋅q_z) ⎥
⎢                                   ⎥
⎢            ⎛      2        2    ⎞ ⎥
⎣a_{b,z} + g⋅⎝- 2⋅qₓ  - 2⋅q_y  + 1⎠ ⎦

## Face detection

\begin{align*}
h_{sq} &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
h_{sp} &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\\
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

In [ ]:
# TODO

---
# Jacobians for the Filter Equations

## State Transition Jacobian $F$

We essentially take the state transition function and do the partial derivatives with respect to the error state.

In [ ]:
F_delta_x_IMU = f_error_state_IMU.jacobian(delta_x_IMU)

F_delta_x_n_IMU = f_error_state_IMU.jacobian(delta_x_n_IMU)
# F_delta_x_n is also just a little bit strange Identity matrix so that could be rewritten to perform faster
# F_delta_x_n = sym.eye(15)[:,3:15]    # yields same result as per 15x15 matrix with accel bias in state but without gravity in state

F_delta_x_IMU

⎡1  0  0  \Delta t     0         0                                             ↪
⎢                                                                              ↪
⎢0  1  0     0      \Delta t     0                                             ↪
⎢                                                                              ↪
⎢0  0  1     0         0      \Delta t                                         ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢0  0  0     1         0         0                  \Delta t⋅((-\tilde{a}_y +  ↪
⎢                                                                              ↪
⎢                                                             ⎛                ↪
⎢0  0  0     0         1         0                   \Delta t⋅⎝(-\tilde{a}_y + ↪
⎢                                                                              ↪
⎢                           

## Measurement Jacobian $H$

We need to take the partial derivative of the measurement function with respect to the error state. To do that, we chain the partial derivative of the measurement function with respect to the true state with the partial derivatives of the true state with respect to the error state:

$$
H = \frac{\partial h}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} = H_x X_{\delta x}
$$

The true state is just the nominal state with the error injected $x_t = x \oplus \delta x$ (special case for quaternions)

In [ ]:
#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :] 

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true)

H_x = h_IMU.jacobian(x_IMU)
X_delta_x = true_state.jacobian(delta_x_IMU)

H_delta_x = H_x * X_delta_x

sym.simplify(H_delta_x)

⎡                                                          ⎛     2     2       ↪
⎢0  0  0  0  0  0                 0                  1.0⋅g⋅⎝- q_w  + qₓ  + q_y ↪
⎢                                                                              ↪
⎢                        ⎛   2     2      2      2⎞                            ↪
⎢0  0  0  0  0  0  1.0⋅g⋅⎝q_w  - qₓ  - q_y  + q_z ⎠                  0         ↪
⎢                                                                              ↪
⎣0  0  0  0  0  0     2.0⋅g⋅(-q_w⋅qₓ - q_y⋅q_z)          2.0⋅g⋅(-q_w⋅q_y + qₓ⋅ ↪

↪ 2      2⎞                                            ⎤
↪   - q_z ⎠  2.0⋅g⋅(q_w⋅qₓ + q_y⋅q_z)  1  0  0  0  0  0⎥
↪                                                      ⎥
↪                                                      ⎥
↪            2.0⋅g⋅(q_w⋅q_y - qₓ⋅q_z)  0  1  0  0  0  0⎥
↪                                                      ⎥
↪ q_z)                  0              0  0  1  0  0  0⎦

---
# debugging

In [ ]:
# quaternion test

q_test = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))
p = sym.Matrix(sym.symbols("x, y, z"))

# axis angle test
result_b = quat_from_axis_angle(p, False)

# skew-symmetric matrix test
skew_symmetric_matrix(p)

⎡0   -z  y ⎤
⎢          ⎥
⎢z   0   -x⎥
⎢          ⎥
⎣-y  x   0 ⎦